# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library. It demonstrates how to load the Croissant schema, review available record sets, extract fields using their `@id` references, and perform exploratory data analysis.

### Dataset Source
The dataset source is described via its Croissant schema URL and provides structured cancer survivor tabular data suitable for biomedical investigation.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment in Colab or fresh environments)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata properties (as an object)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"Published: {meta.datePublished}")
print(f"Version: {meta.version}")

## 2. Data Overview
Review available record sets and fields in the dataset schema, by `@id`.

We enumerate the record sets and their fields, displaying `@id`, `name`, and `description`/`dataType` where available.

In [ ]:
# List available record sets and their @id
print("Available record sets (@id):\n")
for rs in dataset.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(Unnamed)')} | {rs.get('description', '')}")

# List fields for each record set
print("\nFields per record set:")
for rs in dataset.record_sets:
    fields = rs.get('field', [])
    field_list = fields if isinstance(fields, list) else [fields]
    print(f"\nRecord set '@id': {rs['@id']} ({rs.get('name', '')})")
    for fld in field_list:
        # Each field is a dict, or a string @id to lookup
        fld_d = fld if isinstance(fld, dict) else dataset.field_by_id(fld)
        print(f"    - Field @id: {fld_d.get('@id', fld)}; name: {fld_d.get('name','')}; dataType: {fld_d.get('dataType','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entity references use `@id` from the overview above.

Below, we extract the full set of record sets into DataFrames, indexed by their `@id`.

In [ ]:
# Collect @id for each record set
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Record set @ids: {record_sets_ids}")

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Extracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Number of records: {len(df)}")
        print(f"Columns: {df.columns.tolist()}")
    else:
        print(f"No records found for {record_set_id}")

# As example, display columns and preview of the first available record set
if len(dataframes):
    chosen_rs = list(dataframes.keys())[0]
    print(f"\nFields in record set {chosen_rs}: {dataframes[chosen_rs].columns.tolist()}")
    print(dataframes[chosen_rs].head())

## 4. Exploratory Data Analysis (EDA)
We demonstrate simple filtering, normalization, and grouping on a numeric field from the selected record set.

All fields are referenced by `@id`.

In [ ]:
# Pick a record set with data
if len(dataframes):
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Inspect available fields
    print(f"Columns in {record_set_id}: {df.columns.tolist()}")

    # Choose a numeric field by @id (customize as needed)
    # Let's try typical examples: 'age' or 'interval_months', or show all numerics
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        possible_numeric = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'months' in col.lower()]
        numeric_fields = possible_numeric

    print(f"Numeric fields detected: {numeric_fields}")

    if len(numeric_fields):
        numeric_field_id = numeric_fields[0]  # Use the first detected field
        threshold = 10
        # Filter rows where field > threshold
        try:
            filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
        except Exception:
            print(f"Could not convert {numeric_field_id} to numeric, skipping filtering.")
            filtered_df = df.copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        print(filtered_df.head())

        # Normalize selected numeric field
        try:
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        except Exception:
            print(f"Could not normalize {numeric_field_id}, skipping normalization.")

        # Group by a categorical field (select the first string/boolean field apart from the numeric one)
        group_fields = [col for col in df.columns if col != numeric_field_id and (pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]))]
        group_field_id = group_fields[0] if len(group_fields) else None
        if group_field_id:
            print(f"Grouping by field: {group_field_id}")
            try:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(grouped_df.head())
            except Exception:
                print("Could not group by selected group field.")
    else:
        print("No numeric fields found for this record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields, referencing field `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram or pairplot if numeric field available
if len(dataframes):
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        field_id = numeric_fields[0]
        plt.figure(figsize=(6,4))
        sns.histplot(df[field_id].astype(float), kde=True, bins=10)
        plt.title(f"Distribution of {field_id}")
        plt.xlabel(field_id)
        plt.show()

        # If >1 numeric, show pairwise scatter
        if len(numeric_fields) > 1:
            sns.pairplot(df[numeric_fields])
            plt.show()
    else:
        print("No numeric fields to plot.")
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to access and analyze the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library. All data entities—including record sets and fields—were referenced by their unique `@id`. Initial exploration provided insight into available record sets, allowed extraction of tabular data for analysis, and enabled statistical and visual exploration of numeric variables. This approach facilitates reproducible FAIR data workflows and will support further domain-specific analyses.